In [1]:
!c:\Users\hp\AppData\Local\Python\pythoncore-3.11-64\python.exe -m pip install ultralytics opencv-python


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: c:\Users\hp\AppData\Local\Python\pythoncore-3.11-64\python.exe -m pip install --upgrade pip


In [16]:
from ultralytics import YOLO

# YOLOv8 modelini yüklüyoruz
model = YOLO('yolov8n.pt')  

# Sadece Amerika verileriyle eğitim baslatıyoruz
results = model.train(
    data='data.yaml',    
    epochs=100,           
    imgsz=640,           
    batch=8              
)

Ultralytics 8.4.165  Python-3.11.9 torch-2.14.0+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overl

In [ ]:
from ultralytics import YOLO

# YOLOv8 nano modelini yüklüyoruz
model = YOLO('yolov8n.pt')  

# Tüm veri setiyle eğitimi başlatıyoruz 
results = model.train(
    data='data.yaml',    
    epochs=50,          # Hocanın istediği veya ideal epoch sayısı
    imgsz=640,           
    batch=8              
)

In [8]:
import os
import shutil

def us_verilerini_ayikla(veri_tipi):
    img_dir = f'dataset/RDD_SPLIT/{veri_tipi}/images'
    lbl_dir = f'dataset/RDD_SPLIT/{veri_tipi}/labels'
    
    # Klasör yoksa atla (örneğin test klasörü olmayabilir)
    if not os.path.exists(img_dir):
        print(f"{veri_tipi} klasörü bulunamadı, geçiliyor.")
        return

    yeni_img_dir = f'dataset/US_ONLY/{veri_tipi}/images'
    yeni_lbl_dir = f'dataset/US_ONLY/{veri_tipi}/labels'
    
    os.makedirs(yeni_img_dir, exist_ok=True)
    os.makedirs(yeni_lbl_dir, exist_ok=True)
    
    sayac = 0
    for dosya in os.listdir(img_dir):
        if dosya.startswith('United_States'):
            # Resmi kopyala
            shutil.copy(os.path.join(img_dir, dosya), os.path.join(yeni_img_dir, dosya))
            
            # Eşleşen etiket (txt) dosyasını kopyala
            txt_dosya = dosya.rsplit('.', 1)[0] + '.txt'
            src_txt = os.path.join(lbl_dir, txt_dosya)
            if os.path.exists(src_txt):
                shutil.copy(src_txt, os.path.join(yeni_lbl_dir, txt_dosya))
                
            sayac += 1
            
    print(f"-> {veri_tipi} için toplam {sayac} adet United States verisi ayrıldı!")

# Train, val ve varsa test için çalıştıralım:
for asama in ['train', 'val', 'test']:
    us_verilerini_ayikla(asama)

-> train için toplam 3348 adet United States verisi ayrıldı!
-> val için toplam 704 adet United States verisi ayrıldı!
-> test için toplam 753 adet United States verisi ayrıldı!


In [1]:
from ultralytics import YOLO

# Eğittiğimiz modeli yüklüyoruz
model = YOLO('runs/detect/train-3/weights/best.pt')

# Doğrulama (val) verileri üzerindeki başarı testini çalıştırıyoruz
metrics = model.val()

# Elde edilen ana metrikleri yazdıralım:
print(f"Ortalama Kesinlik (Precision): {metrics.box.map50}") # mAP@0.5 genel başarı
print(f"Doğruluk/Başarı (mAP50-95): {metrics.box.map}")

WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\hp\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.89  Python-3.11.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 4.47.6 ms, read: 0.40.6 MB/s, size: 81.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\hp\Desktop\road damage\dataset\US_ONLY\val\labels... 704 images, 0 backgrounds, 16 corrupt: 100% ━━━━━━━━━━━━ 704/704 67.1it/s 10.5s<0

In [12]:
from ultralytics import YOLO

# Eğittiğimiz en iyi modelin ağırlıklarını yüklüyoruz 
# (Yol genellikle runs/detect/train-/weights/best.pt şeklindedir)
model = YOLO('runs/detect/train-2/weights/best.pt')

# Test klasöründen rastgele bir görsel üzerinde tahmin yapalım
# (Kendi test görselinin yolunu buraya yazabilirsin)
results = model('dataset/US_ONLY/val/images/United_States_004279.jpg', show=True)


image 1/1 c:\Users\hp\Desktop\road damage\dataset\US_ONLY\val\images\United_States_004279.jpg: 640x640 2 D20s, 95.4ms
Speed: 5.4ms preprocess, 95.4ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)


In [5]:
import glob

# US_ONLY klasör yolun (eğer genel istiyorsan RDD_SPLIT yapabilirsin)
label_files = glob.glob('dataset/US_ONLY/**/*.txt', recursive=True)

# 0: D00, 1: D10, 2: D20, 3: D40 olmak üzere 4 sınıf
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_labels = 0

for file in label_files:
  with open(file, 'r') as f:
    for line in f:
      parts = line.strip().split()
      if parts:
        class_id = int(parts[0])
        if class_id in class_counts:
          class_counts[class_id] += 1
          total_labels += 1

print('--- SINIF BAZLI ETİKET SAYILARI ---')
print(f'D00 (Longitudinal): {class_counts[0]}')
print(f'D10 (Transverse): {class_counts[1]}')
print(f'D20 (Alligator): {class_counts[2]}')
print(f'D40 (Pothole): {class_counts[3]}')  # İşte burası eklenmiş oldu!
print(f'Toplam Etiket Sayısı: {total_labels}')

--- SINIF BAZLI ETİKET SAYILARI ---
D00 (Longitudinal): 6750
D10 (Transverse): 3295
D20 (Alligator): 834
D40 (Pothole): 0
Toplam Etiket Sayısı: 10879
